In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.retail;

In [0]:
%sql
SHOW SCHEMAS IN workspace;

databaseName
default
information_schema
retail


In [0]:
import random
from datetime import date, timedelta

random.seed(42)

products = [("P001","Laptop",900), ("P002","Phone",600), ("P003","Headphones",80),
            ("P004","Monitor",250), ("P005","Keyboard",45), ("P006","Mouse",25)]
regions = ["North", "South", "East", "West"]

rows = []
for i in range(1, 2001):
    pid, pname, price = random.choice(products)
    d = date(2026, 1, 1) + timedelta(days=random.randint(0, 300))
    # mix of date formats on purpose
    order_date = d.strftime("%Y-%m-%d") if random.random() < 0.8 else d.strftime("%d/%m/%Y")
    qty = str(random.randint(1, 5))
    cust = f"C{random.randint(1, 300):04d}"
    region = random.choice(regions)

    # inject dirty data
    if random.random() < 0.03: cust = None                  # missing customer
    if random.random() < 0.02: qty = "-1"                   # invalid quantity
    if random.random() < 0.02: qty = None                   # missing quantity
    if random.random() < 0.05: region = "  " + region.lower() + " "  # messy text

    rows.append((f"O{i:05d}", order_date, cust, pid, pname, qty, str(price), region))

# add ~50 duplicate orders
rows += random.sample(rows, 50)

cols = ["order_id","order_date","customer_id","product_id","product_name",
        "quantity","unit_price","region"]

# everything stored as strings, like a raw CSV landing in bronze
df = spark.createDataFrame(rows, cols)

df.write.mode("overwrite").saveAsTable("workspace.retail.bronze_sales")
print("Rows written:", df.count())

Rows written: 2050
